In [10]:
from pathlib import Path
import json
import gc
import h5py
import numpy as np
import pandas as pd


In [11]:

# --------------------------------------------------
# INPUT MAT FILES
# --------------------------------------------------

MAT_FILES = [
    Path(r"C:\Users\lokan\Document\DL\project\data\2018-04-12_batchdata_updated_struct_errorcorrect.mat"),
    Path(r"C:\Users\lokan\Document\DL\project\data\2017-05-12_batchdata_updated_struct_errorcorrect.mat"),
    Path(r"C:\Users\lokan\Document\DL\project\data\2017-06-30_batchdata_updated_struct_errorcorrect.mat"),
]

# --------------------------------------------------
# OUTPUT DIRECTORY
# --------------------------------------------------

OUTPUT_DIR = Path("battery_extracted")

# Number of cycles required in each dataset
CYCLE_LIMITS = [10, 30, 100]

# --------------------------------------------------
# OUTPUT CSV COLUMNS
# --------------------------------------------------

CSV_COLUMNS = [
    "cell_idx",
    "batch_date",
    "policy",
    "policy_readable",
    "barcode",
    "channel_id",
    "cycle_life",
    "cycle_index",
    "discharge_dQdV",
    "t",
    "Qc",
    "I",
    "V",
    "T",
    "Qd",
    "Qdlin",
    "Tdlin",
    "summary",
    "Vdlin",
]

# --------------------------------------------------
# CHECK INPUT FILES
# --------------------------------------------------

for mat_file in MAT_FILES:
    if not mat_file.exists():
        raise FileNotFoundError(
            f"File not found: {mat_file.resolve()}"
        )

print("All input files found.")

for file in MAT_FILES:
    print(file.resolve())

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nOutput directory:", OUTPUT_DIR.resolve())

All input files found.
C:\Users\lokan\Document\DL\project\data\2018-04-12_batchdata_updated_struct_errorcorrect.mat
C:\Users\lokan\Document\DL\project\data\2017-05-12_batchdata_updated_struct_errorcorrect.mat
C:\Users\lokan\Document\DL\project\data\2017-06-30_batchdata_updated_struct_errorcorrect.mat

Output directory: C:\Users\lokan\Document\DL\ENDSEM\data\battery_extracted


In [12]:
for mat_file in MAT_FILES:

    print("\n" + "=" * 70)
    print("FILE:", mat_file.name)
    print("=" * 70)

    with h5py.File(mat_file, "r") as f:

        print("Top-level fields:")
        print(list(f.keys()))

        if "batch" not in f:
            raise KeyError(
                f"'batch' not found in {mat_file.name}"
            )

        batch = f["batch"]

        print("\nBatch fields:")
        print(list(batch.keys()))

        for field in batch.keys():

            obj = batch[field]

            if isinstance(obj, h5py.Dataset):
                print(
                    f"{field}: "
                    f"shape={obj.shape}, "
                    f"dtype={obj.dtype}"
                )
            else:
                print(f"{field}: HDF5 group")


FILE: 2018-04-12_batchdata_updated_struct_errorcorrect.mat
Top-level fields:
['#refs#', '#subsystem#', 'batch', 'batch_date']

Batch fields:
['Vdlin', 'barcode', 'channel_id', 'cycle_life', 'cycles', 'policy', 'policy_readable', 'summary']
Vdlin: shape=(46, 1), dtype=object
barcode: shape=(46, 1), dtype=object
channel_id: shape=(46, 1), dtype=object
cycle_life: shape=(46, 1), dtype=object
cycles: shape=(46, 1), dtype=object
policy: shape=(46, 1), dtype=object
policy_readable: shape=(46, 1), dtype=object
summary: shape=(46, 1), dtype=object

FILE: 2017-05-12_batchdata_updated_struct_errorcorrect.mat
Top-level fields:
['#refs#', '#subsystem#', 'batch', 'batch_date']

Batch fields:
['Vdlin', 'barcode', 'channel_id', 'cycle_life', 'cycles', 'policy', 'policy_readable', 'summary']
Vdlin: shape=(46, 1), dtype=object
barcode: shape=(46, 1), dtype=object
channel_id: shape=(46, 1), dtype=object
cycle_life: shape=(46, 1), dtype=object
cycles: shape=(46, 1), dtype=object
policy: shape=(46, 1), d

In [13]:
def dereference(f, value):
    """Resolve an HDF5 reference."""

    if isinstance(value, h5py.Reference):

        if not value:
            return None

        return f[value]

    return value


def decode_matlab_string(f, value):
    """Decode a MATLAB string or character array."""

    value = dereference(f, value)

    if value is None:
        return None

    if isinstance(value, h5py.Dataset):
        arr = np.asarray(value[()])
    else:
        arr = np.asarray(value)

    if arr.size == 0:
        return None

    # MATLAB UTF-16 character arrays
    if arr.dtype.kind == "u" and arr.dtype.itemsize == 2:

        chars = arr.reshape(-1)

        return "".join(
            chr(int(c))
            for c in chars
            if int(c) != 0
        )

    # Byte strings
    if arr.dtype.kind == "S":

        raw = b"".join(arr.reshape(-1).tolist())

        return raw.decode(
            "utf-8",
            errors="replace"
        ).rstrip("\x00")

    # Unicode strings
    if arr.dtype.kind == "U":

        return "".join(
            arr.reshape(-1).tolist()
        ).rstrip("\x00")

    if arr.size == 1:

        item = arr.reshape(-1)[0]

        if isinstance(item, bytes):

            return item.decode(
                "utf-8",
                errors="replace"
            ).rstrip("\x00")

        return str(item)

    return str(arr)


def read_value(f, value):
    """Read a value from an HDF5 object or reference."""

    value = dereference(f, value)

    if value is None:
        return None

    if isinstance(value, h5py.Group):
        return None

    if isinstance(value, h5py.Dataset):
        arr = np.asarray(value[()])
    else:
        arr = np.asarray(value)

    if arr.size == 0:
        return None

    if arr.dtype.kind in ("S", "U"):
        return decode_matlab_string(f, arr)

    return arr


def to_python(value):
    """Convert NumPy values to Python-compatible values."""

    if value is None:
        return None

    if isinstance(value, np.ndarray):

        if value.size == 1:
            return to_python(value.reshape(-1)[0])

        return [
            to_python(item)
            for item in value.reshape(-1)
        ]

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, bytes):

        return value.decode(
            "utf-8",
            errors="replace"
        )

    return value


def to_json(value):
    """Serialize measurement arrays as JSON strings."""

    return json.dumps(
        to_python(value),
        ensure_ascii=False,
        separators=(",", ":"),
        allow_nan=True,
    )


def read_scalar(f, value):
    """Read scalar metadata."""

    result = read_value(f, value)

    if result is None:
        return None

    if isinstance(result, str):
        return result

    arr = np.asarray(result)

    if arr.size == 1:
        return to_python(arr.reshape(-1)[0])

    return to_python(arr)

In [14]:
def get_cycle_group(f, batch, cell_position):
    """Get the cycle structure for one battery."""

    ref = batch["cycles"][cell_position, 0]

    return dereference(f, ref)


def get_summary_group(f, batch, cell_position):
    """Get the summary structure for one battery."""

    ref = batch["summary"][cell_position, 0]

    return dereference(f, ref)


def get_cycle_count(cycles_group):
    """Determine how many cycles are available."""

    if "I" not in cycles_group:

        raise KeyError(
            "The cycle structure does not contain the 'I' field."
        )

    return cycles_group["I"].shape[0]


def read_cycle_field(
    f,
    cycles_group,
    field_name,
    cycle_position,
):
    """Read one measurement field from a specific cycle."""

    if field_name not in cycles_group:
        return None

    dataset = cycles_group[field_name]

    if cycle_position >= dataset.shape[0]:
        return None

    ref = dataset[cycle_position, 0]

    return read_value(f, ref)


def read_summary_for_cycle(f, summary_group, cycle_position):
    """
    Read summary data for a particular cycle.
    Handles HDF5 references and numeric datasets.
    """

    summary = {}

    for field_name in summary_group.keys():

        dataset = summary_group[field_name]

        if isinstance(dataset, h5py.Group):
            summary[field_name] = None
            continue

        if dataset.shape == ():
            summary[field_name] = to_json(
                read_value(f, dataset)
            )
            continue

        # Read the dataset into a NumPy array.
        arr = np.asarray(dataset[()])

        # Handle reference-based summary fields.
        if h5py.check_dtype(ref=dataset.dtype) is not None:

            refs = arr.reshape(-1)

            if cycle_position >= len(refs):
                summary[field_name] = None
                continue

            value = read_value(
                f,
                refs[cycle_position]
            )

            summary[field_name] = to_json(value)

        else:
            # Handle numeric summary fields.
            arr = arr.reshape(-1)

            if cycle_position >= len(arr):
                summary[field_name] = None
            else:
                summary[field_name] = to_json(
                    arr[cycle_position]
                )

    return summary

In [15]:
def extract_battery_rows(
    f,
    batch,
    cell_position,
    batch_date,
    max_cycles=100,
    start_cycle=1,
):
    """
    Extract cycle measurements and metadata for one battery.

    start_cycle is the original 1-based cycle number.
    """

    cycles_group = get_cycle_group(
        f, batch, cell_position
    )

    summary_group = get_summary_group(
        f, batch, cell_position
    )

    available_cycles = get_cycle_count(cycles_group)

    start_position = start_cycle - 1

    if start_position >= available_cycles:
        return [], available_cycles

    end_position = min(
        start_position + max_cycles,
        available_cycles,
    )

    # Battery metadata.
    metadata = {
        "cell_idx": cell_position,
        "batch_date": batch_date,

        "policy": read_scalar(
            f, batch["policy"][cell_position, 0]
        ),

        "policy_readable": decode_matlab_string(
            f, batch["policy_readable"][cell_position, 0]
        ),

        "barcode": decode_matlab_string(
            f, batch["barcode"][cell_position, 0]
        ),

        "channel_id": read_scalar(
            f, batch["channel_id"][cell_position, 0]
        ),

        "cycle_life": read_scalar(
            f, batch["cycle_life"][cell_position, 0]
        ),
    }

    # Battery-level voltage grid.
    vdlin = read_value(
        f, batch["Vdlin"][cell_position, 0]
    )

    cycle_fields = [
        "discharge_dQdV",
        "t",
        "Qc",
        "I",
        "V",
        "T",
        "Qd",
        "Qdlin",
        "Tdlin",
    ]

    rows = []

    for cycle_position in range(
        start_position, end_position
    ):

        row = metadata.copy()

        # Retain original cycle numbering.
        row["cycle_index"] = cycle_position + 1

        for field_name in cycle_fields:

            value = read_cycle_field(
                f,
                cycles_group,
                field_name,
                cycle_position,
            )

            row[field_name] = to_json(value)

        # Store summary values for this cycle.
        try:
            summary = read_summary_for_cycle(
                f,
                summary_group,
                cycle_position,
            )

            row["summary"] = to_json(summary)

        except Exception as error:
            print(
                f"Warning: Summary extraction failed "
                f"for cell {cell_position}, "
                f"cycle {cycle_position + 1}: {error}"
            )

            row["summary"] = None

        row["Vdlin"] = to_json(vdlin)

        rows.append(row)

    return rows, available_cycles

In [16]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_summary = []

for mat_file in MAT_FILES:

    print("\n" + "=" * 70)
    print("Processing:", mat_file.name)
    print("=" * 70)

    date_name = mat_file.stem.split("_")[0]

    # Special handling for 2017-05-12.
    start_cycle = (
        2 if date_name == "2017-05-12" else 1
    )

    print("Starting original cycle:", start_cycle)

    with h5py.File(mat_file, "r") as f:

        if "batch" not in f:
            raise KeyError(
                f"No batch structure found in {mat_file.name}"
            )

        batch = f["batch"]

        required_fields = [
            "Vdlin",
            "barcode",
            "channel_id",
            "cycle_life",
            "cycles",
            "policy",
            "policy_readable",
            "summary",
        ]

        missing = [
            name for name in required_fields
            if name not in batch
        ]

        if missing:
            raise KeyError(
                f"Missing batch fields: {missing}"
            )

        num_cells = batch["cycles"].shape[0]

        print("Number of battery cells:", num_cells)

        if "batch_date" in f:
            batch_date = decode_matlab_string(
                f, f["batch_date"]
            )
        else:
            batch_date = date_name

        all_rows = []

        # Extract a maximum of 100 cycles per battery.
        for cell_position in range(num_cells):

            rows, available_cycles = extract_battery_rows(
                f=f,
                batch=batch,
                cell_position=cell_position,
                batch_date=batch_date,
                max_cycles=100,
                start_cycle=start_cycle,
            )

            all_rows.extend(rows)

            print(
                f"Cell {cell_position}: "
                f"available={available_cycles}, "
                f"extracted={len(rows)}"
            )

        all_data = pd.DataFrame(
            all_rows,
            columns=CSV_COLUMNS,
        )

        # Create the three datasets.
        for cycle_limit in CYCLE_LIMITS:

            output_folder = (
                OUTPUT_DIR / f"cycles_{cycle_limit}"
            )

            output_folder.mkdir(
                parents=True,
                exist_ok=True,
            )

            if all_data.empty:
                output_data = pd.DataFrame(
                    columns=CSV_COLUMNS
                )

            else:
                # Keep the first N extracted cycles
                # for each individual battery.
                output_data = (
                    all_data
                    .sort_values(
                        ["cell_idx", "cycle_index"]
                    )
                    .groupby(
                        "cell_idx",
                        sort=False,
                        group_keys=False,
                    )
                    .head(cycle_limit)
                    .copy()
                )

                output_data = output_data[CSV_COLUMNS]

            output_file = (
                output_folder / f"{date_name}.csv"
            )

            output_data.to_csv(
                output_file,
                index=False,
            )

            print(
                f"\nSaved: {output_file}"
                f"\nRows: {len(output_data)}"
                f"\nCells: {output_data['cell_idx'].nunique()}"
            )

            output_summary.append({
                "file": mat_file.name,
                "cycle_limit": cycle_limit,
                "start_cycle": start_cycle,
                "cells": output_data["cell_idx"].nunique(),
                "rows": len(output_data),
                "output": str(output_file),
            })

            del output_data

        del all_data
        gc.collect()

print("\nEXTRACTION COMPLETED")

display(pd.DataFrame(output_summary))


Processing: 2018-04-12_batchdata_updated_struct_errorcorrect.mat
Starting original cycle: 1
Number of battery cells: 46
Cell 0: available=1008, extracted=100
Cell 1: available=1062, extracted=100
Cell 2: available=1266, extracted=100
Cell 3: available=1114, extracted=100
Cell 4: available=1047, extracted=100
Cell 5: available=827, extracted=100
Cell 6: available=666, extracted=100
Cell 7: available=1835, extracted=100
Cell 8: available=827, extracted=100
Cell 9: available=1038, extracted=100
Cell 10: available=1077, extracted=100
Cell 11: available=816, extracted=100
Cell 12: available=931, extracted=100
Cell 13: available=815, extracted=100
Cell 14: available=857, extracted=100
Cell 15: available=875, extracted=100
Cell 16: available=1637, extracted=100
Cell 17: available=1314, extracted=100
Cell 18: available=1145, extracted=100
Cell 19: available=1154, extracted=100
Cell 20: available=812, extracted=100
Cell 21: available=771, extracted=100
Cell 22: available=1001, extracted=100
Ce

,file,cycle_limit,start_cycle,cells,rows,output
0,2018-04-12_batchdata_updated_struct_errorcorre...,10,1,46,460,battery_extracted\cycles_10\2018-04-12.csv
1,2018-04-12_batchdata_updated_struct_errorcorre...,30,1,46,1380,battery_extracted\cycles_30\2018-04-12.csv
2,2018-04-12_batchdata_updated_struct_errorcorre...,100,1,46,4600,battery_extracted\cycles_100\2018-04-12.csv
3,2017-05-12_batchdata_updated_struct_errorcorre...,10,2,46,460,battery_extracted\cycles_10\2017-05-12.csv
4,2017-05-12_batchdata_updated_struct_errorcorre...,30,2,46,1380,battery_extracted\cycles_30\2017-05-12.csv
5,2017-05-12_batchdata_updated_struct_errorcorre...,100,2,46,4600,battery_extracted\cycles_100\2017-05-12.csv
6,2017-06-30_batchdata_updated_struct_errorcorre...,10,1,48,480,battery_extracted\cycles_10\2017-06-30.csv
7,2017-06-30_batchdata_updated_struct_errorcorre...,30,1,48,1440,battery_extracted\cycles_30\2017-06-30.csv
8,2017-06-30_batchdata_updated_struct_errorcorre...,100,1,48,4800,battery_extracted\cycles_100\2017-06-30.csv


In [17]:
for cycle_limit in CYCLE_LIMITS:

    folder = OUTPUT_DIR / f"cycles_{cycle_limit}"

    print(f"\n{'=' * 60}")
    print(f"FOLDER: {folder}")
    print(f"{'=' * 60}")

    for csv_file in sorted(folder.glob("*.csv")):

        df = pd.read_csv(csv_file)

        print(f"\nFile: {csv_file.name}")
        print("Rows:", len(df))
        print("Battery cells:", df["cell_idx"].nunique())

        if df.empty:
            print("WARNING: No records extracted.")
            continue

        counts = df.groupby("cell_idx")["cycle_index"].agg(
            ["count", "min", "max"]
        )

        print("Minimum cycles per cell:", counts["count"].min())
        print("Maximum cycles per cell:", counts["count"].max())

        if csv_file.stem == "2017-05-12":

            assert (counts["min"] == 2).all(), (
                "Some batteries do not start at original cycle 2."
            )

            print("PASS: Original cycle 1 was excluded.")

        assert (counts["count"] <= cycle_limit).all()

        print("PASS: Cycle-count limit satisfied.")


FOLDER: battery_extracted\cycles_10

File: 2017-05-12.csv
Rows: 460
Battery cells: 46
Minimum cycles per cell: 10
Maximum cycles per cell: 10
PASS: Original cycle 1 was excluded.
PASS: Cycle-count limit satisfied.

File: 2017-06-30.csv
Rows: 480
Battery cells: 48
Minimum cycles per cell: 10
Maximum cycles per cell: 10
PASS: Cycle-count limit satisfied.

File: 2018-04-12.csv
Rows: 460
Battery cells: 46
Minimum cycles per cell: 10
Maximum cycles per cell: 10
PASS: Cycle-count limit satisfied.

FOLDER: battery_extracted\cycles_30

File: 2017-05-12.csv
Rows: 1380
Battery cells: 46
Minimum cycles per cell: 30
Maximum cycles per cell: 30
PASS: Original cycle 1 was excluded.
PASS: Cycle-count limit satisfied.

File: 2017-06-30.csv
Rows: 1440
Battery cells: 48
Minimum cycles per cell: 30
Maximum cycles per cell: 30
PASS: Cycle-count limit satisfied.

File: 2018-04-12.csv
Rows: 1380
Battery cells: 46
Minimum cycles per cell: 30
Maximum cycles per cell: 30
PASS: Cycle-count limit satisfied.

FO

In [18]:
for file_path in MAT_FILES:

    if not file_path.exists():
        continue

    print("\n" + "=" * 80)
    print("FILE:", file_path.name)
    print("=" * 80)

    with h5py.File(file_path, "r") as f:

        if "batch" not in f:
            print("No batch structure found.")
            continue

        batch = f["batch"]

        # Inspect the first battery.
        ref = batch["cycles"][0, 0]

        if not ref:
            print("First battery has no valid cycle reference.")
            continue

        cycles = f[ref]

        print("\nCycle fields:", list(cycles.keys()))

        for field in cycles.keys():
            obj = cycles[field]

            if isinstance(obj, h5py.Dataset):
                print(
                    f"{field}: shape={obj.shape}, "
                    f"dtype={obj.dtype}"
                )

                if obj.size > 0:
                    value = obj[0, 0]

                    if isinstance(value, h5py.Reference) and value:
                        nested = f[value]

                        if isinstance(nested, h5py.Dataset):
                            print(
                                f"    First cycle value: "
                                f"shape={nested.shape}, "
                                f"dtype={nested.dtype}"
                            )
            else:
                print(f"{field}: group")


FILE: 2018-04-12_batchdata_updated_struct_errorcorrect.mat

Cycle fields: ['I', 'Qc', 'Qd', 'Qdlin', 'T', 'Tdlin', 'V', 'discharge_dQdV', 't']
I: shape=(1008, 1), dtype=object
    First cycle value: shape=(1, 762), dtype=float64
Qc: shape=(1008, 1), dtype=object
    First cycle value: shape=(1, 762), dtype=float64
Qd: shape=(1008, 1), dtype=object
    First cycle value: shape=(1, 762), dtype=float64
Qdlin: shape=(1008, 1), dtype=object
    First cycle value: shape=(1, 1000), dtype=float64
T: shape=(1008, 1), dtype=object
    First cycle value: shape=(1, 762), dtype=float64
Tdlin: shape=(1008, 1), dtype=object
    First cycle value: shape=(1, 1000), dtype=float64
V: shape=(1008, 1), dtype=object
    First cycle value: shape=(1, 762), dtype=float64
discharge_dQdV: shape=(1008, 1), dtype=object
    First cycle value: shape=(1, 1000), dtype=float64
t: shape=(1008, 1), dtype=object
    First cycle value: shape=(1, 762), dtype=float64

FILE: 2017-05-12_batchdata_updated_struct_errorcorrect.